# Lab Test Trends Gold Build

Builds `caresync_dbws.gold.lab_test_trends` from the STTM workbook `sttm_silver_to_gold.xlsx`. This notebook follows the documented grain of one row per test type per calendar month and adds month-over-month abnormal-rate trends.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS caresync_dbws.gold;

CREATE OR REPLACE TABLE caresync_dbws.gold.lab_test_trends AS
WITH monthly_base AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date) AS year,
    MONTH(lr.test_date) AS month,
    c.month_name,
    c.quarter_name,
    CAST(COUNT(*) AS INT) AS total_tests,
    CAST(SUM(CASE WHEN lr.result_status = 'Abnormal' THEN 1 ELSE 0 END) AS INT) AS abnormal_test_count,
    CAST(SUM(CASE WHEN lr.result_status = 'Critical' THEN 1 ELSE 0 END) AS INT) AS critical_test_count
  FROM caresync_dbws.silver.lab_results lr
  LEFT JOIN caresync_dbws.silver.calender c
    ON c.datekey = CAST(date_format(date_trunc('month', lr.test_date), 'yyyyMMdd') AS INT)
  GROUP BY
    lr.test_name,
    YEAR(lr.test_date),
    MONTH(lr.test_date),
    c.month_name,
    c.quarter_name
), monthly_rates AS (
  SELECT
    test_name,
    year,
    month,
    month_name,
    quarter_name,
    total_tests,
    abnormal_test_count,
    critical_test_count,
    CAST(ROUND(CASE WHEN total_tests = 0 THEN NULL ELSE (abnormal_test_count * 100.0) / total_tests END, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(CASE WHEN total_tests = 0 THEN NULL ELSE (critical_test_count * 100.0) / total_tests END, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly_base
), monthly_trends AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM monthly_rates
)
SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  CAST(prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CAST(
    ROUND(
      CASE
        WHEN prev_month_abnormal_rate_pct IS NULL THEN NULL
        ELSE abnormal_rate_pct - prev_month_abnormal_rate_pct
      END,
      2
    ) AS DECIMAL(6,2)
  ) AS abnormal_rate_change_pct
FROM monthly_trends;


In [0]:
updated_df = spark.read.table("caresync_dbws.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))
